In [ ]:
import os, sys, subprocess, pathlib
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

assert pathlib.Path.cwd().name == 'Comparisons', 'run this notebook from the Comparisons/ folder'
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
from calculations.figures import results_dir, save_panels
from calculations import sweep_figures as sf
from calculations.permeability import UNITS_SIEVERTS, UNITS_HENRY
FIGS = results_dir("study-comparison")                  # sweep CSVs, figures and fit tables go here

# The studies to compare. Every value used for a study comes from its own study_config.py.
STUDIES = ['Guo_etal_2025', 'Fuerst_et_al_2024', 'incoloy802_cr2o3']

SHORT_NAMES = {   # metal key in a study_config.py -> label on the figures; any other metal shows its key
    '316L_Guo_2025':                                   '316L',
    'Hastelloy_N_fuerst_2024':                         'Hastelloy N',
    'metal_X40_NiCrAlTi_31_19_Incoloy802_Schmidt1985': 'Incoloy 802',
}
REFRESH = False   # True recomputes every study. Otherwise a study is recomputed only when its CSV is
                  # missing or older than its study_config.py, compare_worker.py or the model code.
R = 8.314         # J/(mol K)

In [ ]:
WORKER = pathlib.Path('compare_worker.py').resolve()
sweep_csv = lambda study: FIGS / f'{study}_sweeps.csv'
CODE = [WORKER, *(ROOT / 'calculations').rglob('*.py')]

def stale(study):
    """Recompute when the CSV is missing or older than the study's config, the worker or the model code."""
    csv = sweep_csv(study)
    newest_input = max(p.stat().st_mtime for p in [ROOT / study / 'study_config.py', *CODE])
    return REFRESH or not csv.exists() or csv.stat().st_mtime < newest_input

jobs = {}
for study in STUDIES:                                    # one process per study, all at once
    if stale(study):
        log = open(FIGS / f'{study}_sweeps.log', 'w')
        env = dict(os.environ, PERMEATION_STUDY=str(ROOT / study / 'study_config.py'))
        jobs[study] = (subprocess.Popen([sys.executable, str(WORKER), str(sweep_csv(study))],
                                        cwd=ROOT, env=env, stdout=log, stderr=subprocess.STDOUT), log)
for study, (proc, log) in jobs.items():
    proc.wait(); log.close()
    if proc.returncode:
        raise RuntimeError(f'{study} failed -- see {log.name}')
    print(f'{study}: computed')

data = pd.concat([pd.read_csv(sweep_csv(s)).assign(folder=s) for s in STUDIES], ignore_index=True)
data['label'] = data.metal.map(lambda m: SHORT_NAMES.get(m, m))
if data.groupby('label').folder.nunique().max() > 1:   # two studies on one metal: say which is which
    data['label'] = data.label + ' (' + data.folder + ')'
LABELS = {s: data.loc[data.folder == s, 'label'].iloc[0] for s in STUDIES}

henry = data['level'] == 'L2a'                           # Henry: J·L/ΔP; every other level: J·L/Δ√P
drive = np.where(henry, data.P - data.P_down, np.sqrt(data.P) - np.sqrt(data.P_down))
data['phi'] = data.flux * data.L_total / drive
data['phi_units'] = np.where(henry, UNITS_HENRY, UNITS_SIEVERTS)

# What each study was actually run with, read back from its own results
RUN_WITH = (data.groupby('folder', sort=False)
                [['label', 'metal', 'oxide', 'T_op', 'P_up', 'P_down', 'L_ox0', 'L_m0',
                  'oxide_fp', 'micro_fp', 'defects_fp', 'conditions_fp']].first())
RUN_WITH['ratio'] = RUN_WITH.L_m0 / RUN_WITH.L_ox0
display(RUN_WITH)

def shared(col, text, otherwise):
    """Title text for one setting: its value if every study uses the same one, else the fallback."""
    values = RUN_WITH[col].unique()
    return text(values[0]) if len(values) == 1 else otherwise

SAME_BLOCKS = all(RUN_WITH[c].nunique() == 1 for c in ('oxide_fp', 'micro_fp', 'defects_fp'))
WALL_NOTE = ('same oxide, microstructure and defects' if SAME_BLOCKS
             else 'oxide / microstructure / defects differ by study')

In [ ]:
# Figures: calculations/sweep_figures.py draws every figure of this notebook in the house style, one figure
# per panel (J and Φ_app, regime-resolved), each written by calculations.figures.save_panels as
# <name>-<panel>.pdf, a 600 dpi .png and a _caption.tex whose numbers come from the results. Colour and marker
# mean the regime: colour = the limiting step, an open mark = a defect path carries most of the flux, a
# diamond = trap control (calculations/wall_labels.py); each alloy is named at its curve's end.
from calculations.wall_labels import describe as look, fitted

LEVEL_TITLES = {
    'L1':    'Level 1: Perfect metal (lattice diffusion)',
    'L2a':   'Level 2a: Perfect oxide alone (molecular H$_2$)',
    'L2b':   'Level 2b: Perfect oxide + perfect metal',
    'L3':    'Level 3: Defective oxide + perfect metal',
    'L4':    'Level 4: Defective metal (traps + grain boundaries)',
    'L5':    'Level 5: Defective oxide + defective metal',
    'L1L6':  'Level 1+6: Perfect metal + surface kinetics',
    'L2aL6': 'Level 2a+6: Perfect oxide alone + surface kinetics',
    'L2bL6': 'Level 2b+6: Perfect oxide + perfect metal + surface kinetics',
    'L3L6':  'Level 3+6: Defective oxide + perfect metal + surface kinetics',
    'L4L6':  'Level 4+6: Perfect oxide + defective metal + surface kinetics',
    'L5L6':  'Level 5+6: Defective oxide + defective metal + surface kinetics',
}

In [ ]:
EXPONENT = {'T': 'E_a (kJ/mol)', 'P': 'n (J) / slope (Φ_app)', 'L': 'm (J) / slope (Φ_app)',
            'f': 'n (J, Φ_app ∝ f^n)'}

def fit_table(rows):
    """Every fitted line as one row: shown under the figures, and returned for the CSV."""
    table = pd.DataFrame(rows).replace({r'\$_2\$': '₂'}, regex=True)
    with pd.option_context('display.float_format', '{:.4g}'.format, 'display.width', 220,
                           'display.max_columns', 20):
        display(table)
    return table

def fit(kind, xs, y):
    """kind 'T': Arrhenius on xs = 1000/T, E_a in kJ/mol. Otherwise a power law y = A·xs^k (xs in SI)."""
    if kind == 'T':
        f = stats.linregress(xs, np.log(y))
        return -f.slope * R, np.exp(f.intercept), f.rvalue**2, np.exp(f.intercept + f.slope * xs)
    f = stats.linregress(np.log(xs), np.log(y))
    return f.slope, np.exp(f.intercept), f.rvalue**2, np.exp(f.intercept) * xs**f.slope

def fit_rows(kind, xs, window, y, regime, by_regime, rows, info):
    """One metal's fits, one group (by_regime False: the single fit) or one per regime; every fit is a table row."""
    for reg in (dict.fromkeys(regime) if by_regime else ['all']):
        m = (regime == reg) if by_regime else np.full(len(y), True)
        name = look(reg)[0] if by_regime else 'All points (single fit)'
        if not fitted(reg) or m.sum() < 3:
            continue                                     # drawn, not fitted
        k, A, r2, _ = fit(kind, xs[m], y[m])
        rows.append({**info, 'fit': name if by_regime else 'single fit',
                     'window': f'{window[m].min():.4g}–{window[m].max():.4g}', 'points': int(m.sum()),
                     EXPONENT[kind]: k, 'A (SI)': A, 'R²': r2})

def curves(level, sweep):
    """[(metal label, that metal's rows of this sweep, in order along it), ...]; empty if the layer is absent."""
    d = data[(data.level == level) & (data.sweep == sweep)]
    key = {'temperature': 'T', 'pressure': 'P', 'oxide': 'L_ox', 'metal': 'L_m', 'total': 'L_total', 'pinholes': 'f'}[sweep]
    return [(LABELS[s], g.sort_values(key)) for s in STUDIES if len(g := d[d.folder == s])]

def study_curves(cs, x_col):
    """The curves of one figure for calculations/sweep_figures.py, one per alloy, with each study's own conditions."""
    return [sf.curve(label, d[x_col], d.flux, d.phi, d.regime, d['T'], d.P, d.L_ox, d.L_m) for label, d in cs]

def downstream(cs):
    return np.concatenate([d.P_down.to_numpy() for _, d in cs])

In [ ]:
def sweep_figure(level, sweep):
    """The temperature or pressure sweep of one level, J and Φ_app (one figure each, one curve per alloy,
    regime-resolved fits), and its fit table (single and regime-resolved fits). Returns (panels, fit table)."""
    kind, cs = ('T' if sweep == 'temperature' else 'P'), curves(level, sweep)
    rows = []
    for col_y, qty in (('flux', 'J'), ('phi', 'Phi_app')):
        for by_regime in (False, True):
            for label, d in cs:
                window = d['T' if kind == 'T' else 'P'].to_numpy()
                xs = 1000 / window if kind == 'T' else window
                fit_rows(kind, xs, window, d[col_y].to_numpy(), d.regime.to_numpy(), by_regime, rows,
                         {'quantity': qty, 'metal': label})
    panels = sf.sweep_panels(sweep, level, study_curves(cs, 'T' if kind == 'T' else 'P'),
                             cs[0][1].phi_units.iloc[0], downstream(cs), variant='studies')
    return sf.show(panels), fit_table(rows)

THICKNESS = {'oxide': 'L_ox', 'metal': 'L_m', 'total': 'L_total'}   # sweep: column swept (m)

def thickness_figure(level):
    """The three thickness sweeps of one level, J and Φ_app (one figure each per sweep the level has a layer
    for, one curve per alloy), and their fit table. Returns (panels, fit table)."""
    rows, panels = [], []
    for sweep, x_col in THICKNESS.items():
        cs = curves(level, sweep)
        if not cs:
            continue                                     # the level has no such layer
        for col_y, qty in (('flux', 'J'), ('phi', 'Phi_app')):
            for label, d in cs:
                xs = d[x_col].to_numpy()
                fit_rows('L', xs, xs, d[col_y].to_numpy(), d.regime.to_numpy(), True, rows,
                         {'quantity': qty, 'sweep': sweep, 'metal': label})
        panels += sf.sweep_panels(sf.THICKNESS_KEYS[sweep], level, study_curves(cs, x_col),
                                  cs[0][1].phi_units.iloc[0], downstream(cs), variant='studies', group=sweep)
    return sf.show(panels), fit_table(rows)

def pinhole_figure(level):
    """J and Φ_app against pinhole area fraction (pinholes only; one figure each, one curve per alloy) and
    the fit table. Returns (panels, fit table)."""
    cs = curves(level, 'pinholes')
    rows = []
    for col_y, qty in (('flux', 'J'), ('phi', 'Phi_app')):
        for label, d in cs:
            xs = d['f'].to_numpy()
            fit_rows('f', xs, xs, d[col_y].to_numpy(), d.regime.to_numpy(), True, rows,
                     {'quantity': qty, 'metal': label})
    panels = sf.sweep_panels('pinholes', level, study_curves(cs, 'f'), cs[0][1].phi_units.iloc[0],
                             downstream(cs), variant='studies')
    return sf.show(panels), fit_table(rows)


In [ ]:
for level in LEVEL_TITLES:
    panels, fits = sweep_figure(level, 'temperature')
    save_panels(panels, f"{level}_temperature_by_study", FIGS, module=sf.MODULE)
    fits.to_csv(FIGS / f"{level}_temperature_by_study_fits.csv", index=False)

In [ ]:
for level in LEVEL_TITLES:
    panels, fits = sweep_figure(level, 'pressure')
    save_panels(panels, f"{level}_pressure_by_study", FIGS, module=sf.MODULE)
    fits.to_csv(FIGS / f"{level}_pressure_by_study_fits.csv", index=False)

In [ ]:
for level in LEVEL_TITLES:
    panels, fits = thickness_figure(level)
    save_panels(panels, f"{level}_thickness_by_study", FIGS, module=sf.MODULE)
    fits.to_csv(FIGS / f"{level}_thickness_by_study_fits.csv", index=False)

In [ ]:
for level in [lv for lv in LEVEL_TITLES if ((data.level == lv) & (data.sweep == 'pinholes')).any()]:
    panels, fits = pinhole_figure(level)
    save_panels(panels, f"{level}_pinholes_by_study", FIGS, module=sf.MODULE)
    fits.to_csv(FIGS / f"{level}_pinholes_by_study_fits.csv", index=False)